# PatchCore ViT-B/16 (x64)

Mirrors the main ViT-B/16 PatchCore experiment at 64x64 resolution to complete the resolution
comparison table (WRN50 x64, EffNet-B0 x64 vs. ViT-B/16 x64 vs. ViT-B/16 x224).

At 64x64 the 16-pixel patch size yields (64/16) = 16 patch tokens per image
instead of 196. The positional embeddings are resized automatically by timm.

All other settings (block 6, 95th-pct val-normal threshold, topk=0.10, mb=400k) are
identical to the main x224 run.

Artifacts written by this notebook:
- `experiments/anomaly_detection/patchcore/vit_b16/x64/main/artifacts/checkpoints/`
- `experiments/anomaly_detection/patchcore/vit_b16/x64/main/artifacts/plots/`
- `experiments/anomaly_detection/patchcore/vit_b16/x64/main/artifacts/results/`


## Setup


In [ ]:
import importlib, subprocess, sys
for pkg in ['timm', 'tqdm', 'scikit-learn']:
    if importlib.util.find_spec(pkg.replace('-', '_').split('scikit')[0] if 'scikit' in pkg else pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('deps ready')


## Imports


In [ ]:
import gc, json, os, random, warnings
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support, confusion_matrix
from tqdm.auto import tqdm
from IPython.display import display
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_CUDA = DEVICE.type == 'cuda'
if USE_CUDA:
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
print('Device:', DEVICE)


## Configuration


In [ ]:
try:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = None
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
            PROJECT_ROOT = candidate
            break
    if PROJECT_ROOT is None:
        raise RuntimeError('Could not locate repo root')
    DATA_PATH = str(PROJECT_ROOT / 'data' / 'raw' / 'LSWMD.pkl')
    IMAGE_SIZE = 64
    VIT_FEATURE_BLOCK = 6
    PATCH_EMBED_DIM = 128
    MEMORY_BANK_MAX = 400000
    TOPK_PATCH_RATIO = 0.1
    PATCHCORE_NN_K = 3
    SCORE_CHUNK = 512
    THRESHOLD_QUANTILE = 0.95
    TRAIN_NORMAL_N = 40000
    VAL_NORMAL_N = 5000
    TEST_NORMAL_N = 5000
    TEST_DEFECT_N = 250
    BATCH_SIZE = 256
    NUM_WORKERS = 0
    ARTIFACT_DIR = PROJECT_ROOT / 'experiments/anomaly_detection/patchcore/vit_b16/x64/main/artifacts'
    CHECKPOINTS_DIR = ARTIFACT_DIR / 'checkpoints'
    PLOTS_DIR = ARTIFACT_DIR / 'plots'
    RESULTS_DIR = ARTIFACT_DIR / 'results'
    for d in [CHECKPOINTS_DIR, PLOTS_DIR, RESULTS_DIR]:
        d.mkdir(parents=True, exist_ok=True)
    print(f'Image size      : {IMAGE_SIZE}x{IMAGE_SIZE}')
    print(f'Patch tokens    : {(IMAGE_SIZE // 16) ** 2} per image')
    print(f'ViT block       : {VIT_FEATURE_BLOCK}')
    print(f'Artifacts       : {ARTIFACT_DIR}')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "cwd = path.cwd().resolve()\nproject_root = none\nfor candidate in [cwd, *cwd.parents]:\n    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():\n        project_root = candidate\n        break\nif project_root is none:\n    raise runtimeerror('could not locate repo root')\ndata_path = str(project_root / 'data' / 'raw' / 'lswmd.pkl')\nimage_size = 64\nvit_feature_block = 6\npatch_embed_dim = 128\nmemory_bank_max = 400000\ntopk_patch_ratio = 0.1\npatchcore_nn_k = 3\nscore_chunk = 512\nthreshold_quantile = 0.95\ntrain_normal_n = 40000\nval_normal_n = 5000\ntest_normal_n = 5000\ntest_defect_n = 250\nbatch_size = 256\nnum_workers = 0\nartifact_dir = project_root / 'experiments/anomaly_detection/patchcore/vit_b16/x64/main/artifacts'\ncheckpoints_dir = artifact_dir / 'checkpoints'\nplots_dir = artifact_dir / 'plots'\nresults_dir = artifact_dir / 'results'\nfor d in [checkpoints_dir, plots_dir, results_dir]:\n    d.mkdir(parents=true, exist_ok=true)\nprint(f'image size      : {image_size}x{image_size}')\nprint(f'patch tokens    : {(image_size // 16) ** 2} per image')\nprint(f'vit block       : {vit_feature_block}')\nprint(f'artifacts       : {artifact_dir}')\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Load Dataset


In [ ]:
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from wafer_defect.data.legacy_pickle import read_legacy_pickle
df_raw = read_legacy_pickle(DATA_PATH)

def parse_failure_label(v):
    if v is None:
        return 'unknown'
    if isinstance(v, float) and np.isnan(v):
        return 'unknown'
    if isinstance(v, (list, tuple, np.ndarray)):
        a = np.array(v).reshape(-1)
        return 'unknown' if len(a) == 0 else str(a[0])
    return str(v)
df = df_raw.copy()
df['failure_label'] = df['failureType'].apply(parse_failure_label).str.strip()
invalid = {'0', 'unknown', 'nan', 'None', '[]'}
df = df[~df['failure_label'].isin(invalid)].copy()
df['is_anomaly'] = (df['failure_label'].str.lower() != 'none').astype(int)
normal_df = df[df['is_anomaly'] == 0].copy()
defect_df = df[df['is_anomaly'] == 1].copy()
print(f'Labeled: {len(df):,}  Normal: {len(normal_df):,}  Defect: {len(defect_df):,}')
rng = np.random.default_rng(SEED)
ns = normal_df.iloc[rng.permutation(len(normal_df))].reset_index(drop=True)
ds = defect_df.iloc[rng.permutation(len(defect_df))].reset_index(drop=True)
a = TRAIN_NORMAL_N
b = a + VAL_NORMAL_N
c = b + TEST_NORMAL_N
train_normal_df = ns.iloc[0:a].copy()
val_normal_df = ns.iloc[a:b].copy()
test_normal_df = ns.iloc[b:c].copy()
test_defect_df = ds.iloc[0:TEST_DEFECT_N].copy()
del df_raw, df, normal_df, defect_df, ns, ds
gc.collect()
print(f'Train normal : {len(train_normal_df):>7,}')
print(f'Val normal   : {len(val_normal_df):>7,}')
print(f'Test normal  : {len(test_normal_df):>7,}')
print(f'Test defect  : {len(test_defect_df):>7,}')


## Dataset and Model


In [ ]:
try:

    class WaferDataset(Dataset):

        def __init__(self, frame: pd.DataFrame, size: int):
            self.maps = frame['waferMap'].values
            self.labels = frame['is_anomaly'].values.astype(np.int64)
            self.size = size

        def __len__(self):
            return len(self.maps)

        def __getitem__(self, idx):
            arr = np.clip(np.array(self.maps[idx], dtype=np.int64), 0, 2)
            x = torch.tensor(arr, dtype=torch.long)
            x = F.one_hot(x, num_classes=3).permute(2, 0, 1).float()
            x = F.interpolate(x.unsqueeze(0), size=(self.size, self.size), mode='nearest').squeeze(0)
            return (x, int(self.labels[idx]))
    loader_kw = dict(batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=USE_CUDA, persistent_workers=NUM_WORKERS > 0)
    train_loader = DataLoader(WaferDataset(train_normal_df, IMAGE_SIZE), **loader_kw)
    val_loader = DataLoader(WaferDataset(val_normal_df, IMAGE_SIZE), **loader_kw)
    test_normal_loader = DataLoader(WaferDataset(test_normal_df, IMAGE_SIZE), **loader_kw)
    test_defect_loader = DataLoader(WaferDataset(test_defect_df, IMAGE_SIZE), **loader_kw)
    xb, _ = next(iter(train_loader))
    print(f'Batch shape: {tuple(xb.shape)}')

    class ViTPatchExtractor(nn.Module):

        def __init__(self, block_idx=VIT_FEATURE_BLOCK, proj_dim=PATCH_EMBED_DIM):
            super().__init__()
            self.vit = timm.create_model('vit_base_patch16_224.augreg_in21k_ft_in1k', pretrained=True, num_classes=0, img_size=IMAGE_SIZE)
            self._feat = {}
            self.vit.blocks[block_idx].register_forward_hook(lambda m, i, o: self._feat.__setitem__('out', o))
            self.proj = nn.Linear(768, proj_dim, bias=False)

        def forward(self, x):
            self._feat = {}
            self.vit(x)
            toks = self._feat['out'][:, 1:, :]
            return self.proj(toks)
    extractor = ViTPatchExtractor().to(DEVICE).eval()
    for p in extractor.parameters():
        p.requires_grad = False
    with torch.inference_mode():
        dummy = torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
        out = extractor(dummy)
    print(f'Extractor output: {tuple(out.shape)}  (B, n_tokens, proj_dim)')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "class waferdataset(dataset):\n\n    def __init__(self, frame: pd.dataframe, size: int):\n        self.maps = frame['wafermap'].values\n        self.labels = frame['is_anomaly'].values.astype(np.int64)\n        self.size = size\n\n    def __len__(self):\n        return len(self.maps)\n\n    def __getitem__(self, idx):\n        arr = np.clip(np.array(self.maps[idx], dtype=np.int64), 0, 2)\n        x = torch.tensor(arr, dtype=torch.long)\n        x = f.one_hot(x, num_classes=3).permute(2, 0, 1).float()\n        x = f.interpolate(x.unsqueeze(0), size=(self.size, self.size), mode='nearest').squeeze(0)\n        return (x, int(self.labels[idx]))\nloader_kw = dict(batch_size=batch_size, shuffle=false, num_workers=num_workers, pin_memory=use_cuda, persistent_workers=num_workers > 0)\ntrain_loader = dataloader(waferdataset(train_normal_df, image_size), **loader_kw)\nval_loader = dataloader(waferdataset(val_normal_df, image_size), **loader_kw)\ntest_normal_loader = dataloader(waferdataset(test_normal_df, image_size), **loader_kw)\ntest_defect_loader = dataloader(waferdataset(test_defect_df, image_size), **loader_kw)\nxb, _ = next(iter(train_loader))\nprint(f'batch shape: {tuple(xb.shape)}')\n\nclass vitpatchextractor(nn.module):\n\n    def __init__(self, block_idx=vit_feature_block, proj_dim=patch_embed_dim):\n        super().__init__()\n        self.vit = timm.create_model('vit_base_patch16_224.augreg_in21k_ft_in1k', pretrained=true, num_classes=0, img_size=image_size)\n        self._feat = {}\n        self.vit.blocks[block_idx].register_forward_hook(lambda m, i, o: self._feat.__setitem__('out', o))\n        self.proj = nn.linear(768, proj_dim, bias=false)\n\n    def forward(self, x):\n        self._feat = {}\n        self.vit(x)\n        toks = self._feat['out'][:, 1:, :]\n        return self.proj(toks)\nextractor = vitpatchextractor().to(device).eval()\nfor p in extractor.parameters():\n    p.requires_grad = false\nwith torch.inference_mode():\n    dummy = torch.zeros(2, 3, image_size, image_size, device=device)\n    out = extractor(dummy)\nprint(f'extractor output: {tuple(out.shape)}  (b, n_tokens, proj_dim)')\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Build Memory Bank and Score All Splits


In [ ]:
try:
    scores_path = RESULTS_DIR / 'scores.npz'
    FORCE_RERUN = False
    MODEL_EXPORT_PATH = CHECKPOINTS_DIR / 'model.pt'
    LEGACY_MODEL_EXPORT_PATH = CHECKPOINTS_DIR / 'best_model.pt'
    if scores_path.exists() and (not FORCE_RERUN):
        checkpoint_path = MODEL_EXPORT_PATH if MODEL_EXPORT_PATH.exists() else LEGACY_MODEL_EXPORT_PATH
        if checkpoint_path.exists():
            checkpoint = torch.load(checkpoint_path, map_location='cpu')
            if isinstance(checkpoint, dict) and 'extractor_state_dict' in checkpoint:
                extractor.load_state_dict(checkpoint['extractor_state_dict'])
                print(f'Loaded extractor weights from {checkpoint_path.name}')
        print(f'Loading saved scores from {scores_path}')
        d = np.load(scores_path)
        val_normal_scores_z = d['val_normal_z']
        test_normal_scores_z = d['test_normal_z']
        test_defect_scores_z = d['test_defect_z']
        mu = float(d['train_mu'])
        std = float(d['train_std'])
    elif FORCE_RERUN:
        if FORCE_RERUN:
            if FORCE_RERUN:
                if FORCE_RERUN:

                    def extract_embeddings(xb):
                        with torch.inference_mode():
                            with torch.autocast('cuda', torch.float16, enabled=USE_CUDA):
                                emb = extractor(xb)
                            emb = emb.float().reshape(-1, emb.shape[-1])
                            emb = F.normalize(emb, p=2, dim=1)
                        return emb
                    sampled, ratio = ([], None)
                    for xb, _ in tqdm(train_loader, desc='Bank build', unit='batch'):
                        xb = xb.to(DEVICE)
                        emb = extract_embeddings(xb)
                        if ratio is None:
                            est_total = len(emb) // len(xb) * len(train_normal_df)
                            ratio = min(1.0, MEMORY_BANK_MAX / est_total)
                            print(f'Tokens/image: {len(emb) // len(xb)}  Est total: {est_total:,}  Sample ratio: {ratio:.4f}')
                        if ratio < 1.0:
                            k = max(1, int(round(len(emb) * ratio)))
                            idx = torch.randperm(len(emb), device=DEVICE)[:k]
                            emb = emb[idx]
                        sampled.append(emb)
                    memory_bank = torch.cat(sampled, dim=0)
                    del sampled
                    gc.collect()
                    if len(memory_bank) > MEMORY_BANK_MAX:
                        memory_bank = memory_bank[torch.randperm(len(memory_bank), device=DEVICE)[:MEMORY_BANK_MAX]]
                    memory_bank = F.normalize(memory_bank, p=2, dim=1).contiguous()
                    memory_bank_t = memory_bank.t().contiguous()
                    print(f'Bank: {len(memory_bank):,} x {memory_bank.shape[1]}-d')

                    def min_dist(patches, bank_t, chunk=SCORE_CHUNK, k=PATCHCORE_NN_K):
                        out = []
                        for i in range(0, len(patches), chunk):
                            p = patches[i:i + chunk]
                            sim = p @ bank_t
                            best = sim.topk(min(k, sim.shape[1]), dim=1).values
                            dist = torch.sqrt(torch.clamp(2.0 - 2.0 * best, min=0.0)).mean(dim=1)
                            out.append(dist)
                        return torch.cat(out)

                    def score_loader_fn(loader, desc='Score'):
                        scores = []
                        with torch.inference_mode():
                            for xb, _ in tqdm(loader, total=len(loader), desc=desc, unit='batch'):
                                xb = xb.to(DEVICE)
                                emb = extract_embeddings(xb)
                                ps = min_dist(emb, memory_bank_t)
                                b = len(xb)
                                ps = ps.reshape(b, -1)
                                topk = max(1, min(int(round(ps.shape[1] * TOPK_PATCH_RATIO)), ps.shape[1]))
                                scores.append(ps.topk(topk, dim=1).values.mean(dim=1).cpu())
                        return torch.cat(scores).numpy()
                    train_scores = score_loader_fn(train_loader, 'Score train')
                    val_scores = score_loader_fn(val_loader, 'Score val')
                    test_normal_scores = score_loader_fn(test_normal_loader, 'Score test-normal')
                    test_defect_scores = score_loader_fn(test_defect_loader, 'Score test-defect')
                    mu = float(np.mean(train_scores))
                    std = float(np.std(train_scores) + 1e-08)
                    val_normal_scores_z = (val_scores - mu) / std
                    test_normal_scores_z = (test_normal_scores - mu) / std
                    test_defect_scores_z = (test_defect_scores - mu) / std
                    np.savez_compressed(scores_path, val_normal_z=val_normal_scores_z, test_normal_z=test_normal_scores_z, test_defect_z=test_defect_scores_z, train_mu=np.array(mu), train_std=np.array(std))
                    print(f'Scores saved. mu={mu:.4f}  std={std:.4f}')
                    torch.save({'extractor_state_dict': extractor.state_dict(), 'memory_bank': memory_bank.cpu(), 'config': dict(backbone='vit_base_patch16_224.augreg_in21k_ft_in1k', image_size=IMAGE_SIZE, vit_feature_block=VIT_FEATURE_BLOCK, patch_embed_dim=PATCH_EMBED_DIM, memory_bank_max=MEMORY_BANK_MAX, topk_patch_ratio=TOPK_PATCH_RATIO, patchcore_nn_k=PATCHCORE_NN_K), 'train_mu': mu, 'train_std': std}, MODEL_EXPORT_PATH)
                    del memory_bank, memory_bank_t
                    gc.collect()
                    if USE_CUDA:
                        torch.cuda.empty_cache()
                else:
                    print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
            else:
                print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
        else:
            print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
    else:
        print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
    print(f'val_normal  : n={len(val_normal_scores_z):,}  mean={val_normal_scores_z.mean():.3f}')
    print(f'test_normal : n={len(test_normal_scores_z):,}  mean={test_normal_scores_z.mean():.3f}')
    print(f'test_defect : n={len(test_defect_scores_z):,}  mean={test_defect_scores_z.mean():.3f}')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "scores_path = results_dir / 'scores.npz'\nforce_rerun = false\nmodel_export_path = checkpoints_dir / 'model.pt'\nlegacy_model_export_path = checkpoints_dir / 'best_model.pt'\nif scores_path.exists() and (not force_rerun):\n    checkpoint_path = model_export_path if model_export_path.exists() else legacy_model_export_path\n    if checkpoint_path.exists():\n        checkpoint = torch.load(checkpoint_path, map_location='cpu')\n        if isinstance(checkpoint, dict) and 'extractor_state_dict' in checkpoint:\n            extractor.load_state_dict(checkpoint['extractor_state_dict'])\n            print(f'loaded extractor weights from {checkpoint_path.name}')\n    print(f'loading saved scores from {scores_path}')\n    d = np.load(scores_path)\n    val_normal_scores_z = d['val_normal_z']\n    test_normal_scores_z = d['test_normal_z']\n    test_defect_scores_z = d['test_defect_z']\n    mu = float(d['train_mu'])\n    std = float(d['train_std'])\nelif force_rerun:\n\n    def extract_embeddings(xb):\n        with torch.inference_mode():\n            with torch.autocast('cuda', torch.float16, enabled=use_cuda):\n                emb = extractor(xb)\n            emb = emb.float().reshape(-1, emb.shape[-1])\n            emb = f.normalize(emb, p=2, dim=1)\n        return emb\n    sampled, ratio = ([], none)\n    for xb, _ in tqdm(train_loader, desc='bank build', unit='batch'):\n        xb = xb.to(device)\n        emb = extract_embeddings(xb)\n        if ratio is none:\n            est_total = len(emb) // len(xb) * len(train_normal_df)\n            ratio = min(1.0, memory_bank_max / est_total)\n            print(f'tokens/image: {len(emb) // len(xb)}  est total: {est_total:,}  sample ratio: {ratio:.4f}')\n        if ratio < 1.0:\n            k = max(1, int(round(len(emb) * ratio)))\n            idx = torch.randperm(len(emb), device=device)[:k]\n            emb = emb[idx]\n        sampled.append(emb)\n    memory_bank = torch.cat(sampled, dim=0)\n    del sampled\n    gc.collect()\n    if len(memory_bank) > memory_bank_max:\n        memory_bank = memory_bank[torch.randperm(len(memory_bank), device=device)[:memory_bank_max]]\n    memory_bank = f.normalize(memory_bank, p=2, dim=1).contiguous()\n    memory_bank_t = memory_bank.t().contiguous()\n    print(f'bank: {len(memory_bank):,} x {memory_bank.shape[1]}-d')\n\n    def min_dist(patches, bank_t, chunk=score_chunk, k=patchcore_nn_k):\n        out = []\n        for i in range(0, len(patches), chunk):\n            p = patches[i:i + chunk]\n            sim = p @ bank_t\n            best = sim.topk(min(k, sim.shape[1]), dim=1).values\n            dist = torch.sqrt(torch.clamp(2.0 - 2.0 * best, min=0.0)).mean(dim=1)\n            out.append(dist)\n        return torch.cat(out)\n\n    def score_loader_fn(loader, desc='score'):\n        scores = []\n        with torch.inference_mode():\n            for xb, _ in tqdm(loader, total=len(loader), desc=desc, unit='batch'):\n                xb = xb.to(device)\n                emb = extract_embeddings(xb)\n                ps = min_dist(emb, memory_bank_t)\n                b = len(xb)\n                ps = ps.reshape(b, -1)\n                topk = max(1, min(int(round(ps.shape[1] * topk_patch_ratio)), ps.shape[1]))\n                scores.append(ps.topk(topk, dim=1).values.mean(dim=1).cpu())\n        return torch.cat(scores).numpy()\n    train_scores = score_loader_fn(train_loader, 'score train')\n    val_scores = score_loader_fn(val_loader, 'score val')\n    test_normal_scores = score_loader_fn(test_normal_loader, 'score test-normal')\n    test_defect_scores = score_loader_fn(test_defect_loader, 'score test-defect')\n    mu = float(np.mean(train_scores))\n    std = float(np.std(train_scores) + 1e-08)\n    val_normal_scores_z = (val_scores - mu) / std\n    test_normal_scores_z = (test_normal_scores - mu) / std\n    test_defect_scores_z = (test_defect_scores - mu) / std\n    np.savez_compressed(scores_path, val_normal_z=val_normal_scores_z, test_normal_z=test_normal_scores_z, test_defect_z=test_defect_scores_z, train_mu=np.array(mu), train_std=n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Threshold Selection and Evaluation


In [ ]:
try:
    threshold_z = float(np.quantile(val_normal_scores_z, THRESHOLD_QUANTILE))
    threshold_raw = mu + threshold_z * std
    print(f'Threshold (z)  : {threshold_z:.4f}')
    print(f'Threshold (raw): {threshold_raw:.6f}')
    y_true = np.concatenate([np.zeros(len(test_normal_scores_z), dtype=int), np.ones(len(test_defect_scores_z), dtype=int)])
    scores = np.concatenate([test_normal_scores_z, test_defect_scores_z])
    y_pred = (scores > threshold_z).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    auroc = float(roc_auc_score(y_true, scores))
    auprc = float(average_precision_score(y_true, scores))
    cm = confusion_matrix(y_true, y_pred)
    print(f'\nF1   : {f1:.4f}')
    print(f'AUROC: {auroc:.4f}')
    print(f'AUPRC: {auprc:.4f}')
    print(f'Precision: {p:.4f}  Recall: {r:.4f}')
    print(f'Confusion matrix:\n{cm}')
    metrics = dict(f1=float(f1), auroc=float(auroc), auprc=float(auprc), precision=float(p), recall=float(r), threshold_z=float(threshold_z), threshold_raw=float(threshold_raw), train_mu=float(mu), train_std=float(std), confusion_matrix=cm.tolist(), image_size=IMAGE_SIZE, vit_feature_block=VIT_FEATURE_BLOCK, n_patch_tokens=(IMAGE_SIZE // 16) ** 2)
    (RESULTS_DIR / 'evaluation_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "threshold_z = float(np.quantile(val_normal_scores_z, threshold_quantile))\nthreshold_raw = mu + threshold_z * std\nprint(f'threshold (z)  : {threshold_z:.4f}')\nprint(f'threshold (raw): {threshold_raw:.6f}')\ny_true = np.concatenate([np.zeros(len(test_normal_scores_z), dtype=int), np.ones(len(test_defect_scores_z), dtype=int)])\nscores = np.concatenate([test_normal_scores_z, test_defect_scores_z])\ny_pred = (scores > threshold_z).astype(int)\np, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)\nauroc = float(roc_auc_score(y_true, scores))\nauprc = float(average_precision_score(y_true, scores))\ncm = confusion_matrix(y_true, y_pred)\nprint(f'\\nf1   : {f1:.4f}')\nprint(f'auroc: {auroc:.4f}')\nprint(f'auprc: {auprc:.4f}')\nprint(f'precision: {p:.4f}  recall: {r:.4f}')\nprint(f'confusion matrix:\\n{cm}')\nmetrics = dict(f1=float(f1), auroc=float(auroc), auprc=float(auprc), precision=float(p), recall=float(r), threshold_z=float(threshold_z), threshold_raw=float(threshold_raw), train_mu=float(mu), train_std=float(std), confusion_matrix=cm.tolist(), image_size=image_size, vit_feature_block=vit_feature_block, n_patch_tokens=(image_size // 16) ** 2)\n(results_dir / 'evaluation_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Plots


In [ ]:
try:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(val_normal_scores_z, bins=40, alpha=0.7, color='#457b9d', label='Val normal', density=True)
    axes[0].axvline(threshold_z, color='red', linestyle='--', label=f'={threshold_z:.3f}')
    axes[0].set_title('Val Score Distribution')
    axes[0].set_xlabel('z-score')
    axes[0].set_ylabel('Density')
    axes[0].legend()
    axes[1].hist(test_normal_scores_z, bins=40, alpha=0.6, color='#577590', label='Test normal', density=True)
    axes[1].hist(test_defect_scores_z, bins=40, alpha=0.6, color='#e76f51', label='Test defect', density=True)
    axes[1].axvline(threshold_z, color='red', linestyle='--', label=f'={threshold_z:.3f}')
    axes[1].set_title('Test Score Distribution')
    axes[1].set_xlabel('z-score')
    axes[1].legend()
    plt.suptitle(f'ViT-B/16 PatchCore x64  F1={f1:.3f}  AUROC={auroc:.3f}  AUPRC={auprc:.3f}')
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'score_distribution.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    fig2, ax2 = plt.subplots(figsize=(4.5, 4))
    im = ax2.imshow(cm, cmap='Blues')
    for (row_i, col_i), val in np.ndenumerate(cm):
        ax2.text(col_i, row_i, str(val), ha='center', va='center', color='black')
    ax2.set_xticks([0, 1], ['pred normal', 'pred anomaly'])
    ax2.set_yticks([0, 1], ['true normal', 'true anomaly'])
    ax2.set_title('Confusion Matrix')
    fig2.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
    plt.tight_layout()
    fig2.savefig(PLOTS_DIR / 'confusion_matrix.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig2)
    print(f'Plots saved to {PLOTS_DIR}')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "fig, axes = plt.subplots(1, 2, figsize=(12, 4))\naxes[0].hist(val_normal_scores_z, bins=40, alpha=0.7, color='#457b9d', label='val normal', density=true)\naxes[0].axvline(threshold_z, color='red', linestyle='--', label=f'={threshold_z:.3f}')\naxes[0].set_title('val score distribution')\naxes[0].set_xlabel('z-score')\naxes[0].set_ylabel('density')\naxes[0].legend()\naxes[1].hist(test_normal_scores_z, bins=40, alpha=0.6, color='#577590', label='test normal', density=true)\naxes[1].hist(test_defect_scores_z, bins=40, alpha=0.6, color='#e76f51', label='test defect', density=true)\naxes[1].axvline(threshold_z, color='red', linestyle='--', label=f'={threshold_z:.3f}')\naxes[1].set_title('test score distribution')\naxes[1].set_xlabel('z-score')\naxes[1].legend()\nplt.suptitle(f'vit-b/16 patchcore x64  f1={f1:.3f}  auroc={auroc:.3f}  auprc={auprc:.3f}')\nplt.tight_layout()\nfig.savefig(plots_dir / 'score_distribution.png', dpi=200, bbox_inches='tight')\nplt.show()\nplt.close(fig)\nfig2, ax2 = plt.subplots(figsize=(4.5, 4))\nim = ax2.imshow(cm, cmap='blues')\nfor (row_i, col_i), val in np.ndenumerate(cm):\n    ax2.text(col_i, row_i, str(val), ha='center', va='center', color='black')\nax2.set_xticks([0, 1], ['pred normal', 'pred anomaly'])\nax2.set_yticks([0, 1], ['true normal', 'true anomaly'])\nax2.set_title('confusion matrix')\nfig2.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)\nplt.tight_layout()\nfig2.savefig(plots_dir / 'confusion_matrix.png', dpi=200, bbox_inches='tight')\nplt.show()\nplt.close(fig2)\nprint(f'plots saved to {plots_dir}')\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Per-Defect Recall Breakdown


In [ ]:
try:
    tmp = test_defect_df.copy().reset_index(drop=True)
    tmp['score'] = test_defect_scores_z
    tmp['detected'] = (test_defect_scores_z > threshold_z).astype(int)
    defect_df_out = tmp.groupby('failure_label').agg(count=('detected', 'count'), detected=('detected', 'sum'), recall=('detected', 'mean'), mean_score=('score', 'mean')).reset_index().sort_values('recall', ascending=False)
    display(defect_df_out.round(3))
    defect_df_out.to_csv(RESULTS_DIR / 'defect_breakdown.csv', index=False)
    fig3, ax3 = plt.subplots(figsize=(9, 4))
    plot_df = defect_df_out.sort_values('recall', ascending=True)
    ax3.barh(plot_df['failure_label'], plot_df['recall'], color='#81b29a')
    ax3.set_xlim(0, 1)
    ax3.set_title('Per-Defect Recall - ViT-B/16 PatchCore x64')
    ax3.set_xlabel('Recall')
    plt.tight_layout()
    fig3.savefig(PLOTS_DIR / 'defect_breakdown.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig3)
    print(f"Defect breakdown saved to {RESULTS_DIR / 'defect_breakdown.csv'}")
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = 'tmp = test_defect_df.copy().reset_index(drop=true)\ntmp[\'score\'] = test_defect_scores_z\ntmp[\'detected\'] = (test_defect_scores_z > threshold_z).astype(int)\ndefect_df_out = tmp.groupby(\'failure_label\').agg(count=(\'detected\', \'count\'), detected=(\'detected\', \'sum\'), recall=(\'detected\', \'mean\'), mean_score=(\'score\', \'mean\')).reset_index().sort_values(\'recall\', ascending=false)\ndisplay(defect_df_out.round(3))\ndefect_df_out.to_csv(results_dir / \'defect_breakdown.csv\', index=false)\nfig3, ax3 = plt.subplots(figsize=(9, 4))\nplot_df = defect_df_out.sort_values(\'recall\', ascending=true)\nax3.barh(plot_df[\'failure_label\'], plot_df[\'recall\'], color=\'#81b29a\')\nax3.set_xlim(0, 1)\nax3.set_title(\'per-defect recall - vit-b/16 patchcore x64\')\nax3.set_xlabel(\'recall\')\nplt.tight_layout()\nfig3.savefig(plots_dir / \'defect_breakdown.png\', dpi=200, bbox_inches=\'tight\')\nplt.show()\nplt.close(fig3)\nprint(f"defect breakdown saved to {results_dir / \'defect_breakdown.csv\'}")\n'
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Save Val and Test Score CSVs

Exports score CSVs in the standard format (score, is_anomaly) for downstream ensemble experiments.


In [ ]:
try:
    pd.DataFrame({'score': val_normal_scores_z, 'is_anomaly': np.zeros(len(val_normal_scores_z), dtype=int)}).to_csv(RESULTS_DIR / 'val_scores.csv', index=False)
    pd.DataFrame({'score': np.concatenate([test_normal_scores_z, test_defect_scores_z]), 'is_anomaly': y_true}).to_csv(RESULTS_DIR / 'test_scores.csv', index=False)
    print('val_scores.csv and test_scores.csv saved to', RESULTS_DIR)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "pd.dataframe({'score': val_normal_scores_z, 'is_anomaly': np.zeros(len(val_normal_scores_z), dtype=int)}).to_csv(results_dir / 'val_scores.csv', index=false)\npd.dataframe({'score': np.concatenate([test_normal_scores_z, test_defect_scores_z]), 'is_anomaly': y_true}).to_csv(results_dir / 'test_scores.csv', index=false)\nprint('val_scores.csv and test_scores.csv saved to', results_dir)\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## UMAP Visualization


In [ ]:
try:
    import subprocess
    from sklearn.decomposition import PCA
    UMAP_PNG_PATH = str(globals().get('UMAP_PNG_PATH', Path(PLOTS_DIR) / 'umap_test_embeddings.png'))
    UMAP_CSV_PATH = str(globals().get('UMAP_CSV_PATH', Path(RESULTS_DIR) / 'umap_test_embeddings.csv'))
    seed_value = int(globals().get('SEED', 42))
    if 'test_normal_scores_z' in globals():
        normal_scores = np.asarray(test_normal_scores_z)
        defect_scores = np.asarray(test_defect_scores_z)
        threshold_value = float(globals().get('threshold_z', globals().get('best_thresh')))
        score_column = 'score_z'
    else:
        normal_scores = np.asarray(test_normal_scores)
        defect_scores = np.asarray(test_defect_scores)
        threshold_value = float(globals().get('best_thresh', globals().get('threshold_z')))
        score_column = 'score'
    if 'test_normal_loader' not in globals() or 'test_defect_loader' not in globals():
        batch_size = int(globals().get('BATCH_SIZE', globals().get('BANK_BATCH_SIZE', 128)))
        num_workers = int(globals().get('NUM_WORKERS', globals().get('BANK_WORKERS', 0)))
        _loader_kw = dict(batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=USE_CUDA, persistent_workers=num_workers > 0)
        test_normal_loader = DataLoader(WaferDataset(test_normal_df, IMAGE_SIZE), **_loader_kw)
        test_defect_loader = DataLoader(WaferDataset(test_defect_df, IMAGE_SIZE), **_loader_kw)
    try:
        import umap.umap_ as umap
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn', '-q'])
        import umap.umap_ as umap

    def collect_image_embeddings(loader, frame, split_name, desc):
        embeddings = []
        metadata_rows = []
        seen = 0
        with torch.inference_mode():
            for xb, yb in tqdm(loader, total=len(loader), desc=desc, unit='batch'):
                batch_size = len(yb)
                xb = xb.to(DEVICE)
                with torch.autocast('cuda', torch.float16, enabled=USE_CUDA):
                    feat = extractor(xb)
                    emb = extractor.proj(feat)
                img_emb = F.normalize(emb.float().mean(dim=1), p=2, dim=1)
                embeddings.append(img_emb.cpu().numpy())
                batch_meta = frame.iloc[seen:seen + batch_size][['failure_label', 'is_anomaly']].copy()
                batch_meta = batch_meta.reset_index(drop=True)
                batch_meta['split'] = split_name
                metadata_rows.append(batch_meta)
                seen += batch_size
        return (np.concatenate(embeddings, axis=0), pd.concat(metadata_rows, ignore_index=True))
    X_normal, meta_normal = collect_image_embeddings(test_normal_loader, test_normal_df, 'test_normal', 'Embed test-normal')
    X_defect, meta_defect = collect_image_embeddings(test_defect_loader, test_defect_df, 'test_defect', 'Embed test-defect')
    X = np.concatenate([X_normal, X_defect], axis=0)
    umap_df = pd.concat([meta_normal, meta_defect], ignore_index=True)
    umap_df.insert(0, 'point_index', np.arange(len(umap_df)))
    umap_df[score_column] = np.concatenate([normal_scores, defect_scores])
    umap_df['predicted_anomaly'] = (umap_df[score_column] > threshold_value).astype(int)
    umap_df['label_name'] = umap_df['is_anomaly'].map({0: 'normal', 1: 'defect'})
    n_pca = min(32, X.shape[1], X.shape[0] - 1)
    Xp = PCA(n_components=n_pca, random_state=seed_value).fit_transform(X)
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine', random_state=seed_value, transform_seed=seed_value, low_memory=True)
    coords = reducer.fit_transform(Xp)
    umap_df['umap_1'] = coords[:, 0]
    umap_df['umap_2'] = coords[:, 1]
    fig, ax = plt.subplots(figsize=(8, 6))
    normal_mask = umap_df['is_anomaly'].to_numpy() == 0
    defect_mask = ~normal_mask
    ax.scatter(coords[normal_mask, 0], coords[normal_mask, 1], s=8, alpha=0.45, label='Normal', c='steelblue', linewidths=0)
    ax.scatter(coords[defect_mask, 0], coords[defect_mask, 1], s=8, alpha=0.6, label='Defect', c='tomato', linewidths=0)
    ax.set_title('UMAP of Test Image Embeddings')
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.legend()
    fig.tight_layout()
    fig.savefig(UMAP_PNG_PATH, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    umap_df.to_csv(UMAP_CSV_PATH, index=False)
    print(f'Saved UMAP figure: {UMAP_PNG_PATH}')
    print(f'Saved UMAP points: {UMAP_CSV_PATH}')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "import subprocess\nfrom sklearn.decomposition import pca\numap_png_path = str(globals().get('umap_png_path', path(plots_dir) / 'umap_test_embeddings.png'))\numap_csv_path = str(globals().get('umap_csv_path', path(results_dir) / 'umap_test_embeddings.csv'))\nseed_value = int(globals().get('seed', 42))\nif 'test_normal_scores_z' in globals():\n    normal_scores = np.asarray(test_normal_scores_z)\n    defect_scores = np.asarray(test_defect_scores_z)\n    threshold_value = float(globals().get('threshold_z', globals().get('best_thresh')))\n    score_column = 'score_z'\nelse:\n    normal_scores = np.asarray(test_normal_scores)\n    defect_scores = np.asarray(test_defect_scores)\n    threshold_value = float(globals().get('best_thresh', globals().get('threshold_z')))\n    score_column = 'score'\nif 'test_normal_loader' not in globals() or 'test_defect_loader' not in globals():\n    batch_size = int(globals().get('batch_size', globals().get('bank_batch_size', 128)))\n    num_workers = int(globals().get('num_workers', globals().get('bank_workers', 0)))\n    _loader_kw = dict(batch_size=batch_size, shuffle=false, num_workers=num_workers, pin_memory=use_cuda, persistent_workers=num_workers > 0)\n    test_normal_loader = dataloader(waferdataset(test_normal_df, image_size), **_loader_kw)\n    test_defect_loader = dataloader(waferdataset(test_defect_df, image_size), **_loader_kw)\ntry:\n    import umap.umap_ as umap\nexcept exception:\n    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn', '-q'])\n    import umap.umap_ as umap\n\ndef collect_image_embeddings(loader, frame, split_name, desc):\n    embeddings = []\n    metadata_rows = []\n    seen = 0\n    with torch.inference_mode():\n        for xb, yb in tqdm(loader, total=len(loader), desc=desc, unit='batch'):\n            batch_size = len(yb)\n            xb = xb.to(device)\n            with torch.autocast('cuda', torch.float16, enabled=use_cuda):\n                feat = extractor(xb)\n                emb = extractor.proj(feat)\n            img_emb = f.normalize(emb.float().mean(dim=1), p=2, dim=1)\n            embeddings.append(img_emb.cpu().numpy())\n            batch_meta = frame.iloc[seen:seen + batch_size][['failure_label', 'is_anomaly']].copy()\n            batch_meta = batch_meta.reset_index(drop=true)\n            batch_meta['split'] = split_name\n            metadata_rows.append(batch_meta)\n            seen += batch_size\n    return (np.concatenate(embeddings, axis=0), pd.concat(metadata_rows, ignore_index=true))\nx_normal, meta_normal = collect_image_embeddings(test_normal_loader, test_normal_df, 'test_normal', 'embed test-normal')\nx_defect, meta_defect = collect_image_embeddings(test_defect_loader, test_defect_df, 'test_defect', 'embed test-defect')\nx = np.concatenate([x_normal, x_defect], axis=0)\numap_df = pd.concat([meta_normal, meta_defect], ignore_index=true)\numap_df.insert(0, 'point_index', np.arange(len(umap_df)))\numap_df[score_column] = np.concatenate([normal_scores, defect_scores])\numap_df['predicted_anomaly'] = (umap_df[score_column] > threshold_value).astype(int)\numap_df['label_name'] = umap_df['is_anomaly'].map({0: 'normal', 1: 'defect'})\nn_pca = min(32, x.shape[1], x.shape[0] - 1)\nxp = pca(n_components=n_pca, random_state=seed_value).fit_transform(x)\nreducer = umap.umap(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine', random_state=seed_value, transform_seed=seed_value, low_memory=true)\ncoords = reducer.fit_transform(xp)\numap_df['umap_1'] = coords[:, 0]\numap_df['umap_2'] = coords[:, 1]\nfig, ax = plt.subplots(figsize=(8, 6))\nnormal_mask = umap_df['is_anomaly'].to_numpy() == 0\ndefect_mask = ~normal_mask\nax.scatter(coords[normal_mask, 0], coords[normal_mask, 1], s=8, alpha=0.45, label='normal', c='steelblue', linewidths=0)\nax.scatter(coords[defect_mask, 0], coords[defect_mask, 1], s=8, alpha=0.6, label='defect', c='tomato', linewidths=0)\nax.set_title('umap of test image embeddings')\nax.set_xlabel('umap-1')\nax.set_ylabel('umap-2')\nax.legend()\nfig.tight_layout()\nfig.savefig(um"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Save Outputs


In [ ]:
try:
    import shutil
    checkpoints_dir = Path(globals().get('CHECKPOINTS_DIR', globals().get('CHKPT_DIR', Path(ARTIFACT_DIR) / 'checkpoints')))
    plots_dir = Path(globals().get('PLOTS_DIR', Path(ARTIFACT_DIR) / 'plots'))
    results_dir = Path(globals().get('RESULTS_DIR', Path(ARTIFACT_DIR) / 'results'))
    checkpoints_dir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)
    MODEL_EXPORT_PATH = str(globals().get('MODEL_EXPORT_PATH', checkpoints_dir / 'model.pt'))
    METRICS_EXPORT_PATH = str(globals().get('METRICS_EXPORT_PATH', results_dir / 'evaluation_metrics.json'))
    TEST_SCORES_CSV_PATH = str(globals().get('TEST_SCORES_CSV_PATH', results_dir / 'test_scores.csv'))
    DEFECT_RECALL_CSV_PATH = str(globals().get('DEFECT_RECALL_CSV_PATH', results_dir / 'test_defect_recall.csv'))
    UMAP_PNG_PATH = str(globals().get('UMAP_PNG_PATH', plots_dir / 'umap_test_embeddings.png'))
    UMAP_CSV_PATH = str(globals().get('UMAP_CSV_PATH', results_dir / 'umap_test_embeddings.csv'))
    threshold_value = float(globals().get('threshold_z', globals().get('best_thresh', 0.0)))
    threshold_raw_value = float(globals().get('threshold_raw', threshold_value))
    artifact = {'threshold_z': threshold_value, 'threshold_raw': threshold_raw_value, 'config': {}}
    for key in ['IMAGE_SIZE', 'VIT_FEATURE_BLOCK', 'VIT_FEATURE_BLOCKS', 'PATCH_EMBED_DIM', 'TOPK_PATCH_RATIO', 'PATCHCORE_NN_K']:
        if key in globals():
            artifact['config'][key.lower()] = globals()[key]
    if 'mu' in globals():
        artifact['train_score_mu'] = float(mu)
    if 'std' in globals():
        artifact['train_score_std'] = float(std)
    if 'extractor' in globals():
        artifact['extractor_state_dict'] = extractor.state_dict()
    elif 'model' in globals() and hasattr(model, 'state_dict'):
        artifact['model_state_dict'] = model.state_dict()
    if 'memory_bank' in globals() and memory_bank is not None:
        artifact['memory_bank'] = memory_bank.cpu()
    elif Path(MODEL_EXPORT_PATH).exists():
        try:
            existing_artifact = torch.load(MODEL_EXPORT_PATH, map_location='cpu')
            if isinstance(existing_artifact, dict):
                for key in ['memory_bank', 'extractor_state_dict', 'model_state_dict']:
                    if key in existing_artifact and key not in artifact:
                        artifact[key] = existing_artifact[key]
        except Exception:
            pass
    if artifact:
        torch.save(artifact, MODEL_EXPORT_PATH)
    if 'test_scores_df' in globals():
        test_scores_df.to_csv(TEST_SCORES_CSV_PATH, index=False)
    if 'defect_recall_df' in globals():
        defect_recall_df.to_csv(DEFECT_RECALL_CSV_PATH, index=False)
    elif 'defect_df_out' in globals():
        defect_df_out.rename(columns={'failure_label': 'failure_label'}).to_csv(DEFECT_RECALL_CSV_PATH, index=False)
    elif 'breakdown' in globals():
        breakdown.to_csv(DEFECT_RECALL_CSV_PATH, index=False)
    metrics_payload = {'threshold_z': threshold_value, 'threshold_raw': threshold_raw_value, 'model_export_path': MODEL_EXPORT_PATH, 'test_scores_csv_path': TEST_SCORES_CSV_PATH, 'defect_recall_csv_path': DEFECT_RECALL_CSV_PATH, 'umap_png_path': UMAP_PNG_PATH, 'umap_csv_path': UMAP_CSV_PATH}
    if 'roc_auc' in globals():
        metrics_payload['roc_auc_z'] = float(roc_auc)
    if 'auroc' in globals():
        metrics_payload['roc_auc_z'] = float(auroc)
    if 'cm' in globals():
        metrics_payload['confusion_matrix'] = cm.tolist()
    Path(METRICS_EXPORT_PATH).write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')
    print(f'Saved model artifact: {MODEL_EXPORT_PATH}')
    print(f'Saved metrics: {METRICS_EXPORT_PATH}')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "import shutil\ncheckpoints_dir = path(globals().get('checkpoints_dir', globals().get('chkpt_dir', path(artifact_dir) / 'checkpoints')))\nplots_dir = path(globals().get('plots_dir', path(artifact_dir) / 'plots'))\nresults_dir = path(globals().get('results_dir', path(artifact_dir) / 'results'))\ncheckpoints_dir.mkdir(parents=true, exist_ok=true)\nplots_dir.mkdir(parents=true, exist_ok=true)\nresults_dir.mkdir(parents=true, exist_ok=true)\nmodel_export_path = str(globals().get('model_export_path', checkpoints_dir / 'model.pt'))\nmetrics_export_path = str(globals().get('metrics_export_path', results_dir / 'evaluation_metrics.json'))\ntest_scores_csv_path = str(globals().get('test_scores_csv_path', results_dir / 'test_scores.csv'))\ndefect_recall_csv_path = str(globals().get('defect_recall_csv_path', results_dir / 'test_defect_recall.csv'))\numap_png_path = str(globals().get('umap_png_path', plots_dir / 'umap_test_embeddings.png'))\numap_csv_path = str(globals().get('umap_csv_path', results_dir / 'umap_test_embeddings.csv'))\nthreshold_value = float(globals().get('threshold_z', globals().get('best_thresh', 0.0)))\nthreshold_raw_value = float(globals().get('threshold_raw', threshold_value))\nartifact = {'threshold_z': threshold_value, 'threshold_raw': threshold_raw_value, 'config': {}}\nfor key in ['image_size', 'vit_feature_block', 'vit_feature_blocks', 'patch_embed_dim', 'topk_patch_ratio', 'patchcore_nn_k']:\n    if key in globals():\n        artifact['config'][key.lower()] = globals()[key]\nif 'mu' in globals():\n    artifact['train_score_mu'] = float(mu)\nif 'std' in globals():\n    artifact['train_score_std'] = float(std)\nif 'extractor' in globals():\n    artifact['extractor_state_dict'] = extractor.state_dict()\nelif 'model' in globals() and hasattr(model, 'state_dict'):\n    artifact['model_state_dict'] = model.state_dict()\nif 'memory_bank' in globals() and memory_bank is not none:\n    artifact['memory_bank'] = memory_bank.cpu()\nelif path(model_export_path).exists():\n    try:\n        existing_artifact = torch.load(model_export_path, map_location='cpu')\n        if isinstance(existing_artifact, dict):\n            for key in ['memory_bank', 'extractor_state_dict', 'model_state_dict']:\n                if key in existing_artifact and key not in artifact:\n                    artifact[key] = existing_artifact[key]\n    except exception:\n        pass\nif artifact:\n    torch.save(artifact, model_export_path)\nif 'test_scores_df' in globals():\n    test_scores_df.to_csv(test_scores_csv_path, index=false)\nif 'defect_recall_df' in globals():\n    defect_recall_df.to_csv(defect_recall_csv_path, index=false)\nelif 'defect_df_out' in globals():\n    defect_df_out.rename(columns={'failure_label': 'failure_label'}).to_csv(defect_recall_csv_path, index=false)\nelif 'breakdown' in globals():\n    breakdown.to_csv(defect_recall_csv_path, index=false)\nmetrics_payload = {'threshold_z': threshold_value, 'threshold_raw': threshold_raw_value, 'model_export_path': model_export_path, 'test_scores_csv_path': test_scores_csv_path, 'defect_recall_csv_path': defect_recall_csv_path, 'umap_png_path': umap_png_path, 'umap_csv_path': umap_csv_path}\nif 'roc_auc' in globals():\n    metrics_payload['roc_auc_z'] = float(roc_auc)\nif 'auroc' in globals():\n    metrics_payload['roc_auc_z'] = float(auroc)\nif 'cm' in globals():\n    metrics_payload['confusion_matrix'] = cm.tolist()\npath(metrics_export_path).write_text(json.dumps(metrics_payload, indent=2), encoding='utf-8')\nprint(f'saved model artifact: {model_export_path}')\nprint(f'saved metrics: {metrics_export_path}')\n"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise
